# Embedding Generation & Similarity Testing

We generate embeddings for product titles/descriptions and test retrieval.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from utils.data_loader import load_product_data
from embeddings.text_embeddings import TextEmbedder
from vector_store.vector_db import FAISSVectorStore

# Load data and embedder
df = load_product_data()
embedder = TextEmbedder()

In [ ]:
# Generate embeddings for all products (or a sample)
texts = df['combined_text'].astype(str).tolist()
embeddings = embedder.embed_texts(texts, batch_size=64)   # uses the method we defined
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# Reduce to 2D for visualization using PCA
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(embeddings)
plt.figure(figsize=(10, 8))
plt.scatter(emb_2d[:,0], emb_2d[:,1], alpha=0.6, c=df['price'] if 'price' in df else 'blue')
plt.colorbar(label='Price' if 'price' in df else '')
plt.title("2D PCA Projection of Product Embeddings")
plt.show()

### Test FAISS Similarity Search

In [ ]:
# Build FAISS index (in-memory for testing)
from vector_store.vector_db import FAISSVectorStore
metadata = df.to_dict(orient='records')
ids = df['id'].tolist()
vector_store = FAISSVectorStore(dimension=embeddings.shape[1])
vector_store.build_index(embeddings, metadata, ids)

# Pick a random product and search for similar ones
query_idx = np.random.randint(0, len(df))
query_emb = embeddings[query_idx:query_idx+1]
results = vector_store.search(query_emb, k=5)

print(f"Query product: {df.iloc[query_idx]['title']}")
print("Similar products:")
for res in results:
    print(f"  - {res['metadata']['title']} (distance: {res['distance']:.4f})")

In [ ]:
# Manual text query
query_text = "blue jeans"
query_emb = embedder.model.encode([query_text])[0]
results = vector_store.search(query_emb, k=5)
print(f"Query: '{query_text}'")
for res in results:
    print(f"  - {res['metadata']['title']} (dist: {res['distance']:.4f})")

In [ ]:
# Compare with a different query
query_text = "formal black shoes"
query_emb = embedder.model.encode([query_text])[0]
results = vector_store.search(query_emb, k=5)
print(f"Query: '{query_text}'")
for res in results:
    print(f"  - {res['metadata']['title']} (dist: {res['distance']:.4f})")